In [0]:
# PySpark Configuration for big data processing
spark.conf.set("spark.sql.shuffle.partitions", "200")
print("Spark version:", spark.version)
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

display(dbutils.fs.ls("/Volumes/workspace/default/museum/"))

Spark version: 4.1.0
Shuffle partitions: 200


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/museum/museum dataset.zip,museum dataset.zip,1197224745,1772540753000
dbfs:/Volumes/workspace/default/museum/unzipped/,unzipped/,0,1775745274833


In [0]:
%sh
unzip -o "/Volumes/workspace/default/museum/museum dataset.zip" \
-d "/Volumes/workspace/default/museum/unzipped/"

Archive:  /Volumes/workspace/default/museum/museum dataset.zip
  inflating: /Volumes/workspace/default/museum/unzipped/resource.csv  
  inflating: /Volumes/workspace/default/museum/unzipped/manifest.json  


In [0]:
from pyspark.sql.functions import col, sum as spark_sum, when

csv_path = "/Volumes/workspace/default/museum/unzipped/resource.csv"

df = (spark.read
      .option("header", "true")
      .option("inferSchema", "true")
      .option("multiLine", "true")
      .option("escape", "\"")
      .csv(csv_path))

# Data validation through ingestion
total_rows = df.count()
total_cols = len(df.columns)
print(f"Total rows: {total_rows:,}")
print(f"Total columns: {total_cols}")

# Check null counts for key columns
from pyspark.sql.functions import col, sum as spark_sum, when

key_cols = ["kingdom", "continent", "country", "basisOfRecord", "collectionCode", "year"]
null_counts = df.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in key_cols
])
display(null_counts)

# Save as Parquet partitioned by collectionCode
# Parquet chosen as it works well with Tabelau
parquet_path = "/Volumes/workspace/default/museum/parquet/"
df.write.mode("overwrite").partitionBy("collectionCode").parquet(parquet_path)
print(f"Parquet saved to: {parquet_path}")

# Save as Delta table
df.write.mode("overwrite").saveAsTable("workspace.default.museum_data")
print("Delta table saved")

Total rows: 6,152,101
Total columns: 143


kingdom,continent,country,basisOfRecord,collectionCode,year
5008663,2111352,2187371,0,0,4044540


Parquet saved to: /Volumes/workspace/default/museum/parquet/
Delta table saved


In [0]:
# Verify save
parquet_df = spark.read.parquet(parquet_path)


# Verify Parquet
parquet_df = spark.read.parquet("/Volumes/workspace/default/museum/parquet/")


# Verify Delta table
display(spark.table("workspace.default.museum_data").limit(10))

Parquet row count verified: 6,152,101
Parquet row count: 6,152,101


_id,associatedMedia._id,associatedMedia.assetID,associatedMedia.category,associatedMedia.created,associatedMedia.creator,associatedMedia.format,associatedMedia.identifier,associatedMedia.license,associatedMedia.modified,associatedMedia.PixelXDimension,associatedMedia.PixelYDimension,associatedMedia.rightsHolder,associatedMedia.title,associatedMedia.type,associatedMediaCount,barcode,basisOfRecord,bed,catalogNumber,catalogueDescription,chondriteAchondrite,chronostratigraphy,class,clutchSize,collectionCode,collectionKind,collectionName,commodity,continent,coordinateUncertaintyInMeters,country,created,currentScientificName,dateRegistered,day,decimalLatitude,decimalLongitude,depositType,determinationFiledAs,determinationNames,donorName,earliestAgeOrLowestStage,earliestEonOrLowestEonothem,earliestEpochOrLowestSeries,earliestEraOrLowestErathem,earliestPeriodOrLowestSystem,expedition,exsiccata,exsiccataNumber,extractionMethod,family,fieldNumber,formation,gbifID,gbifIssue,genus,geodeticDatum,georeferenceProtocol,group,habitat,higherClassification,higherGeography,highestBiostratigraphicZone,identificationAsRegistered,identificationDescription,identificationOther,identificationQualifier,identificationVariety,identifiedBy,individualCount,infraspecificEpithet,institutionCode,island,islandGroup,kindOfCollection,kindOfObject,kingdom,latestAgeOrHighestStage,latestEonOrHighestEonothem,latestEpochOrHighestSeries,latestEraOrHighestErathem,latestPeriodOrHighestSystem,lifeStage,lithostratigraphy,locality,lowestBiostratigraphicZone,maximumDepthInMeters,maximumElevationInMeters,member,meteoriteClass,meteoriteGroup,meteoriteType,mine,mineralComplex,minimumDepthInMeters,minimumElevationInMeters,modified,month,observedWeight,occurrence,occurrenceID,occurrenceStatus,order,otherCatalogNumbers,partType,petrologySubtype,petrologyType,phylum,plantDescription,populationCode,preparations,preparationType,preservative,project,recordedBy,recordNumber,recovery,recoveryDate,recoveryWeight,registrationCode,resuspendedIn,samplingProtocol,scientificName,scientificNameAuthorship,setMark,sex,specificEpithet,stateProvince,subDepartment,subfamily,subgenus,suborder,superfamily,taxonRank,texture,totalVolume,typeStatus,verbatimLatitude,verbatimLongitude,vessel,waterBody,year
1884884,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,013038612,PreservedSpecimen,null,1890.1.14.21,null,null,null,null,null,ZOO,null,null,null,null,null,null,2011-02-24T16:41:54.000Z,null,null,null,null,null,null,null,Glyphodon tristis,Professor A C. Haddon,null,null,null,null,null,null,null,null,null,Elapidae,null,null,1056624868,MODIFIED_DATE_INVALID | INSTITUTION_MATCH_FUZZY | INSTITUTION_COLLECTION_MISMATCH,Glyphodon,null,null,null,null,Elapidae,null,null,null,null,null,null,null,null,1,null,NHMUK,Murray Island,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2022-01-21T15:45:11.000Z,null,null,null,8e606ca4-f489-4f86-86c4-b507eb875fb2,present,null,NHMUK:ecatalogue:1884884,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,Glyphodon tristis,null,null,Male,tristis,null,LS Reptiles,null,null,null,null,null,null,null,null,null,null,null,null,null
6691461,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,010060061,PreservedSpecimen,null,NHMUK010060061,null,null,null,Insecta,null,BMNH(E),null,null,null,Europe,70.0,United Kingdom,2016-07-18T20:04:02.000Z,null,null,13,51.450303,-0.892586,null,null,"Bombus terrestris (Linnaeus, 1758)",Bristol University,null,null,null,null,null,null,null,null,null,Apidae,null,null,1825791753,MODIFIED_DATE_INVALID | GEODETIC_DATUM_ASSUMED_WGS84 | INSTITUTION_MATCH_FUZZY | COLLECTION_MATCH_NONE,Bombus,null,null,null,null,Arthropoda; Insecta; Hymenoptera; Apoidea; Apidae; Apinae,Europe; United Kingdom; England; Berkshire,null,null,null,null,null,null,Mr Philip M. Pavett,1,null,NHMUK,null,null,null,null,null,null,null,nul